# Mục 6.4 — Thực hành đánh giá và tối ưu mô hình

Sinh tu ban thao giao trinh. Chay tuan tu tu tren xuong; moi o phu thuoc cac o truoc no.


## 6.4. Thực hành

Mục này tổng hợp các kỹ thuật ở Mục 6.1 đến 6.3 vào một bài thực hành hoàn chỉnh: đánh giá đầy đủ một mô hình, phát hiện quá khớp, xử lý quá khớp bằng cắt tỉa, áp dụng chính quy hóa L1 và L2, tối ưu siêu tham số, và đóng gói thành pipeline triển khai. Công cụ trong thân mục là Python với scikit-learn, vì toàn bộ bảy phần đều cần chạy lại nhiều cấu hình và ghi số một cách tự động. Người học muốn làm cùng bài này trên giao diện đồ họa của WEKA đọc tệp 04-thuc-hanh-chuong-6.md trong thư mục weka của kho học phần (Mục A.3): tệp đó hướng dẫn từng bước với J48, Logistic và CVParameterSelection, kèm khối kết quả nguyên văn để đối chiếu.

## Chuẩn bị môi trường và tập dữ liệu

### Tài nguyên đi kèm

Các ví dụ trong mục này chỉ cần scikit-learn, pandas, scipy cùng optuna, joblib và flask cho hai phần cuối; cách cài đặt và chọn môi trường làm việc xem Phụ lục B. Tập Wisconsin có sẵn trong scikit-learn. Tập Ljubljana là một tệp ARFF — định dạng dữ liệu của WEKA, mô tả trong tệp 01-su-dung-weka.md của thư mục weka (Mục A.3) — tải trực tiếp theo đường dẫn:

**Kết quả đối chiếu**

```
https://raw.githubusercontent.com/FITHOUVN/Machine-Learning-Introduction/main/datasets/weka/breast-cancer.arff
```

Tệp này là bản sao của tập dữ liệu đi kèm bộ cài WEKA do nhóm Waikato phân phối; không cần cài WEKA để dùng nó. Kiểm tra tệp và phiên bản thư viện trước khi chạy, và ghi lại phiên bản cùng kết quả, vì giá trị mặc định của một số tham số thay đổi theo phiên bản scikit-learn.

### Hai tập dữ liệu và vai trò của mỗi tập

Mục 6.4 dùng hai tập dữ liệu khác nhau, mỗi tập giữ một vai trò riêng; đây không phải hai cách giải cùng một bài toán và không so sánh điểm giữa hai tập.

- Phần 6.4.A đến 6.4.C dùng Ljubljana (breast-cancer.arff, 286 mẫu, 9 thuộc tính danh

mục và một cột nhãn; nhãn no-recurrence-events có 201 mẫu, nhãn recurrence-events có 85 mẫu, tức lớp cần phát hiện chỉ chiếm 29,7%). Đây là một bài toán khó và mất cân bằng rõ. Chính vì khó nên nó mới minh họa được luận điểm cốt lõi của Mục 6.1: Accuracy 73% nghe thì ổn, nhưng Recall của lớp tái phát chỉ 0,282, nghĩa là mô hình bỏ sót 61 trong 85 ca. Tập này cũng cho minh chứng quá khớp rất mạnh: cây không giới hạn độ phức tạp chênh tới 33,07 điểm phần trăm giữa tập huấn luyện và kiểm chứng chéo.

- Phần 6.4.D đến 6.4.G dùng Wisconsin Diagnostic (load_breast_cancer, 569 mẫu, 30 đặc

trưng số; 212 mẫu ác tính và 357 mẫu lành tính). Chính quy hóa, Grid Search, Optuna và pipeline triển khai cần đặc trưng số: chính quy hóa L1 và L2 tác động lên hệ số của mô hình tuyến tính nên đi kèm bước chuẩn hóa, còn pipeline triển khai cần một mô hình nhận vào vectơ số để minh họa việc gọi qua API.

Hai tập cùng thuộc chủ đề ung thư vú nhưng khác nguồn, khác đặc trưng và khác nhãn dự đoán: Ljubljana dự đoán tái phát, Wisconsin phân biệt u lành và u ác. Trên Wisconsin, hầu hết chỉ số của các mô hình thông dụng đều ở mức rất cao, nên tập này phù hợp để tập trung vào kỹ thuật tinh chỉnh hơn là để bàn về hạn chế của Accuracy.

### Nạp tập Ljubljana

Mọi thuộc tính của Ljubljana đều là danh mục, nên sau khi đọc tệp ARFF phải giải mã chuỗi và mã hóa one-hot trước khi đưa vào cây quyết định.



In [ ]:
import io
from urllib.request import urlopen
import pandas as pd
from scipy.io import arff

URL = ('https://raw.githubusercontent.com/FITHOUVN/'
       'Machine-Learning-Introduction/main/datasets/weka/breast-cancer.arff')

with urlopen(URL) as f:                      # hoac mo tep .arff da tai ve
    noi_dung = f.read().decode('utf-8')
du_lieu, meta = arff.loadarff(io.StringIO(noi_dung))

bang = pd.DataFrame(du_lieu)
# scipy tra ve gia tri danh muc duoi dang bytes: giai ma sang chuoi
for cot in bang.columns:
    bang[cot] = bang[cot].map(
        lambda v: v.decode() if isinstance(v, bytes) else v)

cot_nhan = meta.names()[-1]        # thuoc tinh cuoi cung la nhan
y = bang[cot_nhan].to_numpy(dtype=object)
X = bang.drop(columns=[cot_nhan])  # 9 thuoc tinh danh muc
DUONG = 'recurrence-events'        # lop duong: ca tai phat
print(X.shape)
print(pd.Series(y).value_counts().to_dict())


## 6.4.A. Đánh giá đầy đủ một mô hình (Mục 6.1)

### Mục tiêu

Huấn luyện một cây quyết định trên Ljubljana, đánh giá bằng kiểm chứng chéo 10 phần, rồi đọc đầy đủ các chỉ số của Mục 6.1 thay vì chỉ đọc Accuracy.

### Một bẫy phải xử lý trước khi chạy: cắt tỉa không bật sẵn

Cây quyết định J48 của WEKA cắt tỉa theo mặc định, còn DecisionTreeClassifier của scikit-learn thì không: mặc định của nó là cây mọc đến khi mọi lá đều thuần nhất. Nếu lấy mặc định của scikit-learn làm cấu hình chuẩn cho phần này, kết quả không phải là một mô hình tạm dùng được mà là một cây quá khớp: Accuracy kiểm chứng chéo tụt còn 65,02% (số liệu ở Phần 6.4.B), và cả luận điểm của mục — "Accuracy nghe có vẻ ổn nhưng lớp thiểu số thì không" — không còn chỗ dựa. Vì vậy phần này cắt tỉa tường minh bằng ccp_alpha = 0.01; ý nghĩa của tham số này và cách chọn giá trị cho nó là nội dung của Phần 6.4.C.

### Mã nguồn



In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (classification_report, confusion_matrix,
                             cohen_kappa_score)

# Moi thuoc tinh deu la danh muc nen phai ma hoa one-hot.
# Gia tri khuyet trong tep ARFF la chuoi '?' nen duoc giu nguyen
# thanh MOT HANG MUC RIENG khi ma hoa (xem ghi chu ben duoi).
ma_hoa = OneHotEncoder(handle_unknown='ignore')


def cay(**tham_so):
    # ccp_alpha phai dat tuong minh: scikit-learn khong cat tia san
    return make_pipeline(ma_hoa,
                         DecisionTreeClassifier(random_state=42, **tham_so))


nhan = ['no-recurrence-events', DUONG]
du_doan = cross_val_predict(cay(ccp_alpha=0.01), X, y, cv=10)

print(confusion_matrix(y, du_doan, labels=nhan))
print(classification_report(y, du_doan, labels=nhan, digits=3))
print('Kappa:', round(cohen_kappa_score(y, du_doan), 4))


### Giá trị khuyết và cách xử lý ở đây

Tệp ARFF ghi giá trị khuyết bằng dấu ?, và loadarff trả về đúng chuỗi đó, nên trong đoạn mã trên '?' không được điền mà được giữ nguyên thành một hạng mục riêng khi mã hóa one-hot: thuộc tính node-caps có 8 ô ? và breast-quad có 1 ô, mỗi thuộc tính vì thế có thêm một cột chỉ báo. Đây đúng là Cách 3 trong sáu cách xử lý dữ liệu thiếu ở Mục 2.3.1 — dùng một hằng số toàn cục, thay mọi giá trị thiếu bằng cùng một nhãn. Với dữ liệu toàn danh mục, cách này hợp lý và gần như không tốn công: bản thân việc "không ghi nhận được" có thể mang thông tin, chẳng hạn khi một xét nghiệm không được chỉ định vì hồ sơ bệnh án đã rõ. Mặt khác, cần nhớ đúng rủi ro mà Mục 2.3.1 nêu kèm Cách 3: mô hình có thể coi ? là một khái niệm có nghĩa và chia nhánh theo nó, trong khi nguyên nhân khuyết chỉ là sơ sót ghi chép. Một lựa chọn khác là điền ? bằng giá trị phổ biến nhất của thuộc tính — tức imputation bằng mode, cũng mô tả ở Mục 2.3.1 — và lựa chọn đó cho kết quả khác, nên khi báo cáo phải nói rõ đang dùng cách nào. Mọi con số in trong các Phần 6.4.A đến 6.4.C đều tính theo cách thứ nhất, tức giữ ? thành một hạng mục riêng.

Hai Phần 6.4.A và 6.4.B chỉ ước lượng hiệu năng của một cấu hình cố định, không có bước lựa chọn nào, nên dùng kiểm chứng chéo trên toàn bộ dữ liệu. Từ Phần 6.4.E trở đi có lựa chọn siêu tham số nên phải tách riêng tập kiểm tra trước khi tìm kiếm, đúng nguyên tắc nêu ở Mục 6.2.1. Tham số cv=10 áp dụng kiểm chứng chéo phân tầng không xáo trộn, nên không cần đặt hạt giống ngẫu nhiên cho bước chia dữ liệu; random_state=42 chỉ để cố định cách cây xử lý các phép chia có điểm bằng nhau.

### Kết quả (Ljubljana, cây quyết định ccp_alpha = 0.01, kiểm chứng chéo 10 phần)

Ma trận nhầm lẫn gộp dự đoán của mười fold:

### Đọc kết quả

Accuracy 73,08% là con số dễ gây yên tâm sai. Nhìn xuống từng lớp thì bức tranh khác: với lớp tái phát, Recall chỉ 0,282, tức mô hình bỏ sót 61 trong 85 ca thật sự tái phát. Trong bối cảnh y tế, đây là loại sai lầm đắt nhất: một ca tái phát bị xếp vào nhóm không tái phát thì bệnh nhân không được theo dõi tiếp. Precision của cùng lớp đó là 0,600, nghĩa là trong 40 ca mô hình gắn nhãn tái phát thì 24 ca đúng; F1 chỉ 0,384 vì Recall quá thấp kéo trung bình điều hòa xuống. Đối chiếu với lớp đa số (Recall 0,920) thấy rõ mô hình nghiêng hẳn về phía lớp đông, đúng hiện tượng mà Mục 6.1 cảnh báo khi dữ liệu mất cân bằng.

Kappa là hệ số đồng thuận của Cohen, đo mức trùng khớp giữa dự đoán và nhãn thật sau khi trừ phần trùng khớp xảy ra do ngẫu nhiên. Giá trị 0,2393 nằm gần 0 hơn gần 1, cho biết phần dự đoán đúng "hơn mức ngẫu nhiên" là khá mỏng — một cách đọc độc lập với Accuracy và dẫn tới cùng kết luận. Dữ liệu mất cân bằng có thể góp phần vào kết quả này, nhưng cần phân tích thêm trước khi kết luận nguyên nhân.

### Lưu ý cách đọc classification_report

Bảng do classification_report in ra có ba dòng tổng hợp. Dòng accuracy là Accuracy chung. Dòng macro avg lấy trung bình cộng không trọng số của hai lớp, nên lớp thiểu số có tiếng nói bằng lớp đa số. Dòng weighted avg lấy trung bình theo số mẫu mỗi lớp; riêng cột Recall của dòng này luôn bằng Accuracy, vì tổng các dự đoán đúng của từng lớp chia cho tổng số mẫu chính là Accuracy. Vì vậy đừng coi weighted avg là bằng chứng mô hình tốt ở mọi lớp.

Cũng cần nói rõ đang báo Accuracy theo cách nào. Ma trận nhầm lẫn gộp dự đoán của mười fold cho 209/286 = 73,08%, còn bình quân mười điểm Accuracy của từng fold — cách cross_validate báo, dùng ở Phần 6.4.B — cho 73,09%. Khoảng lệch ở hàng phần trăm thứ hai là bình thường: 286 không chia hết cho 10 nên các fold không bằng nhau về số mẫu, bình quân theo fold đặt trọng số hơi khác với tỷ lệ gộp.

### Bổ sung chỉ số xếp hạng

Các chỉ số trên đều tính tại ngưỡng quyết định mặc định 0,5. Để có thêm một cách đánh giá không phụ thuộc ngưỡng, tính AUC theo Mục 6.1.3 từ xác suất dự đoán:



In [ ]:
from sklearn.metrics import roc_auc_score

xac_suat = cross_val_predict(cay(ccp_alpha=0.01), X, y, cv=10,
                             method='predict_proba')[:, 1]
# cot 1 ung voi 'recurrence-events' (cac lop xep theo thu tu chu cai)
print('ROC AUC:',
      round(roc_auc_score((y == DUONG).astype(int), xac_suat), 4))


Ghi lại giá trị thực tế khi chạy. Cần đọc con số này một cách thận trọng: cây sau cắt tỉa chỉ còn rất ít lá nên xác suất dự đoán chỉ nhận một số ít giá trị phân biệt, đường ROC vì thế gồm ít điểm gấp khúc chứ không mượt như hình minh họa ở Mục 6.1.3.

## 6.4.B. Phát hiện quá khớp và thiếu khớp (Mục 6.2.1)

### Mục tiêu

So sánh điểm trên tập huấn luyện với điểm kiểm chứng chéo của cùng một cấu hình, theo đúng dấu hiệu quá khớp mô tả ở Mục 6.2.1.

### Mã nguồn



In [ ]:
from sklearn.model_selection import cross_validate

cau_hinh = [('cay khong gioi han', {}),
            ('min_samples_leaf=10', dict(min_samples_leaf=10)),
            ('ccp_alpha=0.01', dict(ccp_alpha=0.01))]

for ten, tham_so in cau_hinh:
    diem = cross_validate(cay(**tham_so), X, y, cv=10, scoring='accuracy',
                          return_train_score=True)
    tr = diem['train_score'].mean() * 100
    te = diem['test_score'].mean() * 100
    print(f'{ten:20s} train {tr:6.2f}%  CV {te:6.2f}%  chenh {tr - te:5.2f}')


### Đối chiếu kết quả

Cột thứ hai là bình quân điểm trên phần huấn luyện của từng fold, không phải điểm của một cây huấn luyện trên toàn bộ 286 mẫu.

### Đọc kết quả

Cây không giới hạn nhớ gần trọn tập huấn luyện (98,10%) nhưng chỉ đạt 65,02% khi kiểm chứng chéo: chênh 33,07 điểm phần trăm, đúng dấu hiệu quá khớp mà Mục 6.2.1 mô tả và cùng cỡ với ví dụ 26 điểm ở Mục 6.2.1. Nguyên nhân thấy ngay khi đếm lá: huấn luyện trên toàn bộ dữ liệu, cây không giới hạn có 94 lá cho 286 mẫu, trung bình chỉ khoảng ba mẫu một lá (xem bảng ở Phần 6.4.C) — mô hình đã khớp tới từng nhóm nhỏ của tập huấn luyện.

Điều quan trọng hơn con số chênh lệch: Accuracy cao trên tập huấn luyện không dùng để chọn mô hình được. Cây không giới hạn hơn cây cắt tỉa 21,41 điểm trên tập huấn luyện (98,10% so với 76,69%) nhưng kém 8,07 điểm khi kiểm chứng chéo (65,02% so với 73,09%). Hai cách hạn chế độ phức tạp đều thu hẹp khoảng chênh và đồng thời tăng điểm kiểm chứng chéo, tức ở đây việc giảm độ phức tạp không phải đánh đổi mà là cải thiện thực sự.

Cũng cần nhớ nhận xét ở Mục 6.2.1: không có ngưỡng chênh lệch cố định đúng cho mọi bài toán. Chênh 3,60 điểm của cấu hình cuối cho biết mô hình ổn định hơn, chứ không bảo đảm đó là cấu hình tốt nhất; phải xem đồng thời mức điểm, các chỉ số theo lớp và độ biến thiên qua các fold. Chiều ngược lại — cắt tỉa quá mạnh làm mô hình quá đơn giản, dẫn tới thiếu khớp — được khảo ở Phần 6.4.C.

## 6.4.C. Xử lý quá khớp bằng cắt tỉa cây (Mục 6.2.1)

### Cắt tỉa trong scikit-learn

DecisionTreeClassifier hỗ trợ hai nhóm cách hạn chế độ phức tạp. Nhóm thứ nhất là cắt tỉa trước: chặn cây lớn lên bằng max_depth, min_samples_split, min_samples_leaf. Nhóm thứ hai là cắt tỉa sau: cho cây mọc xong rồi bỏ dần các nhánh ít đóng góp, điều khiển bằng ccp_alpha. Tham số ccp_alpha đặt một cái giá cho mỗi lá: một nhánh chỉ được giữ nếu phần lỗi nó giảm được lớn hơn cái giá đó. Vì vậy ccp_alpha = 0 nghĩa là không cắt tỉa, và càng tăng ccp_alpha thì cây càng nhỏ.

### Mã nguồn



In [ ]:
for alpha in (0.0, 0.005, 0.01, 0.02, 0.05):
    mo_hinh = cay(ccp_alpha=alpha)
    acc = cross_validate(mo_hinh, X, y, cv=10,
                         scoring='accuracy')['test_score'].mean()
    du_doan = cross_val_predict(mo_hinh, X, y, cv=10)
    cm = confusion_matrix(y, du_doan, labels=nhan)
    recall = cm[1, 1] / cm[1].sum()
    mo_hinh.fit(X, y)          # huan luyen tren toan bo du lieu de dem la
    print(f'{alpha:.3f}  acc {100 * acc:5.2f}%  recall {recall:.3f}'
          f'  so la {mo_hinh[-1].get_n_leaves()}')


### Kết quả dự kiến (Ljubljana, kiểm chứng chéo 10 phần)

Cột "Số lá" đếm trên cây huấn luyện bằng toàn bộ 286 mẫu, còn Accuracy và Recall lấy từ mười fold. Hai cách đo khác nhau giải thích một chỗ dễ gây thắc mắc: ở mức 0,05, cây huấn luyện trên toàn bộ dữ liệu chỉ còn một lá — tức luôn dự đoán lớp đa số — nhưng Recall gộp từ các fold vẫn khác 0 (0,094, tương ứng 8 trong 85 ca), nghĩa là trên một số fold mức phạt đó vẫn chưa đủ để xóa hết phép chia.

### Đọc kết quả

Cắt tỉa đưa Accuracy kiểm chứng chéo từ 65,02% lên 73,09%, tăng 8,07 điểm phần trăm, đồng thời thu nhỏ cây từ 94 lá còn 3 lá. Nhưng tăng mức cắt tỉa không cải thiện vô hạn: qua mốc 0,01, Accuracy giảm dần (71,01% rồi 67,88%) và số lá tiến về 1. Mức 0,05 là một mô hình gần như không học gì — dấu hiệu thiếu khớp theo Mục 6.2.1, chiều ngược lại của Phần 6.4.B. Nói cách khác, có một vùng giá trị hợp lý ở giữa, và vùng đó phải tìm bằng số liệu chứ không suy từ nguyên tắc.

Bảng trên còn cho một bài học thứ hai, quan trọng hơn: chọn theo Accuracy và chọn theo Recall dẫn tới hai câu trả lời khác nhau. Mức 0,01 cho Accuracy cao nhất (73,09%) nhưng Recall lớp tái phát chỉ 0,282, tức phát hiện 24 trong 85 ca. Mức 0,005 có Accuracy thấp hơn 1,06 điểm (72,03%) nhưng Recall 0,412, tức phát hiện 35 ca và bỏ sót 50 thay vì 61. Nếu mục tiêu của bài toán là hạn chế bỏ sót ca tái phát thì mức 0,005 đáng chọn hơn, dù bảng Accuracy nói ngược lại. Đây chính là lý do Mục 6.1 yêu cầu chọn chỉ số theo hậu quả của từng loại sai lầm trước khi so sánh mô hình.

### Giới hạn của cách làm trong phần này

Bảng trên là một thí nghiệm minh họa quan hệ giữa mức cắt tỉa và hiệu năng, không phải quy trình chọn mô hình đúng chuẩn. Năm mức ccp_alpha đều được so bằng cùng mười fold của cùng một tập dữ liệu, nên điểm tốt nhất trong bảng đã bị dùng để ra quyết định và không còn là ước lượng khách quan cho dữ liệu mới. Quy trình đúng là tách tập kiểm tra trước, tìm siêu tham số bằng kiểm chứng chéo trên phần còn lại, rồi đánh giá một lần trên tập kiểm tra — nội dung của Phần 6.4.E.

## 6.4.D. Áp dụng chính quy hóa L1, L2 (Mục 6.2.2)

### Chuyển sang tập Wisconsin

Từ phần này đến hết Mục 6.4, các ví dụ chạy trên Wisconsin Diagnostic (569 mẫu, 30 đặc trưng số, 212 mẫu ác tính và 357 mẫu lành tính). Lý do: chính quy hóa L1 và L2 tác động lên hệ số của mô hình tuyến tính, nên cần đặc trưng số cùng một bước chuẩn hóa để các hệ số so sánh được với nhau; chín thuộc tính danh mục của Ljubljana sau khi mã hóa one-hot chỉ còn là các biến chỉ báo 0/1, ít phù hợp để minh họa vai trò của chuẩn hóa và của việc co hệ số. Không so sánh điểm của phần này với các Phần 6.4.A đến 6.4.C.

Trong toàn bộ mã Python chạy trên Wisconsin, nhãn được mã hóa lại thành 1 = ác tính, 0 = lành tính. Vì vậy Precision, Recall, F1 nhị phân và predict_proba[:, 1] đều hướng đến việc phát hiện ác tính.

### Ghi chú về tham số C: nghịch đảo của cường độ phạt

LogisticRegression của scikit-learn không nhận trực tiếp hệ số λ của Mục 6.2.2 mà nhận tham số C, là nghịch đảo của cường độ phạt: C càng nhỏ thì chính quy hóa càng mạnh, C càng lớn thì càng yếu. Đây cũng là quy ước của tham số C trong SVM đã nêu ở Mục 6.2.2. Hệ quả thực hành: khi dò C trên thang logarit 0,001; 0,01; 0,1; 1; 10; 100 thì chiều tác động ngược với chiều của λ, đầu nhỏ của thang mới là đầu phạt mạnh. Nhầm chiều là lỗi thường gặp — tăng C với ý định "chính quy hóa mạnh hơn" sẽ nhận được một mô hình quá khớp hơn. Điểm mà cross_val_score in ra dưới đây là Accuracy, vì đó là chỉ số mặc định cho mô hình phân lớp.

### So sánh L1, L2 và ElasticNet



In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

X, y_goc = load_breast_cancer(return_X_y=True)
y = (y_goc == 0).astype(int)   # 1: ac tinh; 0: lanh tinh

# l1_ratio = 0 la L2 thuan, = 1 la L1 thuan, o giua la ElasticNet.
# (Tham so penalty= cu bi danh dau loai bo tu scikit-learn 1.8,
#  go han o ban 1.10; dung l1_ratio thay the)
pipe_l2 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(l1_ratio=0.0, C=1.0, max_iter=5000))
])

pipe_l1 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(l1_ratio=1.0, C=1.0,
                              solver='saga', max_iter=5000, random_state=42))
])

pipe_elastic = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(l1_ratio=0.5, C=1.0,
                              solver='saga', max_iter=5000, random_state=42))
])

for ten, pipe in [('L2', pipe_l2), ('L1', pipe_l1),
                  ('ElasticNet', pipe_elastic)]:
    diem = cross_val_score(pipe, X, y, cv=10)
    pipe.fit(X, y)
    khac_0 = int((pipe['lr'].coef_ != 0).sum())
    print(f'{ten:11s} acc {diem.mean():.3f}   so he so khac 0: {khac_0}/30')


### Đối chiếu kết quả

Chạy mã để so sánh ba dạng chính quy hóa trên cùng các fold và cùng cách mã hóa lớp dương, rồi điền vào bảng dưới đây.

Trên tập dữ liệu này, cả ba dạng thường cho Accuracy xấp xỉ nhau và đều ở mức cao; chênh lệch giữa chúng nằm ở số đặc trưng được giữ lại chứ không ở Accuracy. Ghi phiên bản thư viện, điểm thực tế và số hệ số khác 0 khi báo cáo: L1 có thể tạo nghiệm thưa, nhưng số hệ số được giữ phụ thuộc C và phụ thuộc dữ liệu, không phải một con số cố định. Nếu muốn thấy rõ tác dụng của chính quy hóa, giảm C xuống 0,01 rồi chạy lại và so số hệ số khác 0 — theo ghi chú ở trên, đó là chiều phạt mạnh.

## 6.4.E. Tối ưu siêu tham số bằng Grid Search (Mục 6.3.1)

### Thiết kế thí nghiệm

Khác với ví dụ ở Mục 6.3.1, phần thực hành này tìm siêu tham số cho cây quyết định chứ không phải Random Forest, để tiếp nối mạch kiểm soát độ phức tạp của cây ở hai Phần 6.4.B và 6.4.C; số fold là 10 cho khớp với các phần trước. Chỉ số chấm điểm là F1 chứ không phải Accuracy, vì lớp cần phát hiện (ác tính, 212 mẫu) là lớp ít hơn, và theo Mục 6.1 thì F1 phản ánh chất lượng phát hiện lớp đó tốt hơn Accuracy. Khác với các Phần 6.4.A đến 6.4.C, ở đây có bước lựa chọn nên tập kiểm tra phải được tách ra trước khi tìm kiếm và chỉ dùng một lần ở cuối.



In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (GridSearchCV, train_test_split,
                                     StratifiedKFold)
from sklearn.tree import DecisionTreeClassifier

X, y_goc = load_breast_cancer(return_X_y=True)
y = (y_goc == 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
param_grid = {
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 5, 10]}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42),
                    param_grid, cv=cv, scoring='f1', n_jobs=1)
grid.fit(X_train, y_train)
print('Best params:', grid.best_params_)
print('Best CV F1:', round(grid.best_score_, 4))


### Đối chiếu kết quả

Kết quả được in bởi đoạn mã trên; ghi cả best_params_ và best_score_ cùng phiên bản môi trường. Điểm tốt nhất trong quá trình tìm kiếm là điểm kiểm chứng chéo dùng để chọn mô hình, chưa phải ước lượng độc lập trên tập kiểm tra.

Để kết luận cấu hình tối ưu có cải thiện, so sánh với mô hình mặc định trên cùng các fold của phần huấn luyện; sau khi khóa lựa chọn, đánh giá một lần trên tập kiểm tra.

Có 6 × 4 × 3 = 72 cấu hình và 10 fold, tương ứng 720 lần fit trong kiểm chứng chéo, cộng một lần refit cấu hình tốt nhất. Thời gian phụ thuộc máy, dữ liệu và cách chạy song song.

## 6.4.F. Tối ưu siêu tham số bằng Bayesian Optimization (Mục 6.3.2)

### Thực hành với Optuna

Ví dụ sau dùng TPE của Optuna để thử 30 cấu hình trên cùng DecisionTreeClassifier, cùng không gian tham số và cùng các fold với Phần 6.4.E. Đây là minh họa ngân sách tìm kiếm; không mặc định Optuna luôn cho điểm tốt hơn. Vì không gian ở đây là tập giá trị rời rạc lấy lại từ Phần 6.4.E, ví dụ này chưa thể hiện ưu thế của Bayesian Optimization đối với siêu tham số liên tục đã nêu ở Mục 6.3.2; mục đích là so sánh hai ngân sách tìm kiếm trên cùng một không gian.



In [ ]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
# Dung X_train, y_train, cv va param_grid cua Phan E


def objective(trial):
    params = {name: trial.suggest_categorical(name, values)
              for name, values in param_grid.items()}
    model = DecisionTreeClassifier(random_state=42, **params)
    return cross_val_score(model, X_train, y_train, cv=cv,
                           scoring='f1', n_jobs=1).mean()


study = optuna.create_study(direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, n_jobs=1)
print('Best params:', study.best_params)
print('Best CV F1:', round(study.best_value, 4))


### Đối chiếu kết quả

Ghi best_params và best_value thực tế khi chạy. Cùng một cấu hình có thể được đề xuất nhiều lần; 30 trial không nhất thiết tương ứng 30 cấu hình phân biệt.

Với 30 trial và kiểm chứng chéo 10 phần, Optuna dùng 300 lần fit; Grid Search dùng 720 lần fit cho 72 cấu hình. So sánh điểm kiểm chứng chéo và thời gian trong đúng môi trường này, không so 30 trial với 720 lần fit. Chỉ đánh giá mô hình đã chọn trên tập kiểm tra một lần sau khi kết thúc lựa chọn.

## 6.4.G. Đóng gói thành Pipeline triển khai

Bước cuối là đóng gói toàn bộ tiền xử lý và mô hình vào một pipeline.

### Lợi ích của pipeline

Tránh rò rỉ dữ liệu giữa tập huấn luyện và tập kiểm tra; dễ tái sử dụng và triển khai; đảm bảo các bước được áp dụng đúng thứ tự; tích hợp được với GridSearchCV để tối ưu toàn bộ pipeline.

### Mã pipeline hoàn chỉnh

Pipeline gồm hai bước: chuẩn hóa và mô hình phân lớp. GridSearchCV bọc bên ngoài pipeline để tối ưu siêu tham số của bước mô hình, nhờ đó phép chuẩn hóa được học lại trong từng fold và không gây rò rỉ dữ liệu.



In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.datasets import load_breast_cancer

# Nap duoi dang DataFrame de mo hinh NHO TEN COT (feature_names_in_)
du_lieu = load_breast_cancer(as_frame=True)
X = du_lieu.data
y = (du_lieu.target == 0).astype(int)  # 1: ac tinh; 0: lanh tinh
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

# Tao pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

# Tham so cho Grid Search (luu y prefix 'rf__')
param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [5, 10, 20, None]
}

grid = GridSearchCV(pipeline, param_grid, cv=10,
                    scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

# Mo hinh cuoi: pipeline da toi uu
best_pipeline = grid.best_estimator_

# Danh gia tren test set (chua tung dung)
from sklearn.metrics import classification_report
y_pred = best_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))


### Lưu mô hình

Sau khi có pipeline tối ưu, lưu lại để triển khai:



In [ ]:
import joblib

# X_train, y_train va best_pipeline lay tu doan ma o tren
# GridSearchCV da refit best_estimator_ tren toan bo train

# Luu pipeline
joblib.dump(best_pipeline, 'breast_cancer_model.joblib')

# Tai pipeline (o moi truong khac)
loaded_model = joblib.load('breast_cancer_model.joblib')
new_data = X_test[:3]        # 3 mau moi; thuc te lay tu he thong khac
predictions = loaded_model.predict(new_data)
print(predictions)


Lưu được mô hình mới chỉ là nửa việc. Muốn hệ thống khác dùng được mô hình — chẳng hạn phần mềm quản lý học tập gọi sang để hỏi một người học có nguy cơ hay không — thì phải bọc mô hình lại thành một dịch vụ nhận yêu cầu qua mạng. Cách phổ biến nhất là REST API. REST API cung cấp các điểm truy cập (endpoint) để nhận dữ liệu và trả kết quả dự đoán. Dữ liệu trao đổi thường có định dạng JSON, đã giới thiệu ở Mục 2.2.1.

Thư viện Flask cho phép xây dựng dịch vụ bằng vài chục dòng mã. Tệp app.py dưới đây nạp mô hình đã lưu và cung cấp hai endpoint: /health để kiểm tra trạng thái dịch vụ; /predict để dự đoán.



In [ ]:
from flask import Flask, request, jsonify
import joblib
import numpy as np
import pandas as pd
app = Flask(__name__)
mo_hinh = joblib.load('breast_cancer_model.joblib')
TEN_COT = list(mo_hinh.feature_names_in_)
TEN_NHAN = {0: 'lanh_tinh', 1: 'ac_tinh'}


@app.route('/health')
def health():
    return jsonify({'trang_thai': 'san sang'})


@app.route('/predict', methods=['POST'])
def predict():
    du_lieu = request.get_json(silent=True)
    if not isinstance(du_lieu, dict):
        return jsonify({'loi': 'Can mot doi tuong JSON'}), 400
    thieu = [c for c in TEN_COT if c not in du_lieu]
    if thieu:
        return jsonify({'loi': 'Thieu cot', 'cot_thieu': thieu}), 400
    try:
        X = pd.DataFrame([du_lieu])[TEN_COT].astype(float)
        if not np.isfinite(X.to_numpy()).all():
            raise ValueError('Gia tri khong huu han')
    except (TypeError, ValueError):
        return jsonify({'loi': 'Cac cot phai la so huu han'}), 400
    nhan = int(mo_hinh.predict(X)[0])
    probs = mo_hinh.predict_proba(X)[0]
    xac_suat = {TEN_NHAN[int(c)]: round(float(p), 4)
                for c, p in zip(mo_hinh.classes_, probs)}
    return jsonify({'nhan': nhan, 'ten_nhan': TEN_NHAN[nhan],
                    'xac_suat_tung_lop': xac_suat})


if __name__ == '__main__':
    app.run(port=5000)  # may chu phat trien, chi de thuc hanh


Thực thi đoạn mã máy chủ và đoạn gọi thử trong hai cửa sổ lệnh riêng. Ở cửa sổ thứ nhất, nhập python app.py; thông báo Running on http://127.0.0.1:5000 cho biết dịch vụ sẵn sàng chờ yêu cầu. Giữ cửa sổ này mở và thực thi đoạn gọi thử ở cửa sổ thứ hai. Khi kết thúc, nhấn Ctrl+C trong cửa sổ thứ nhất để dừng máy chủ.



In [ ]:
import requests
from sklearn.datasets import load_breast_cancer

print(requests.get('http://127.0.0.1:5000/health').json())
# {'trang_thai': 'san sang'}

# Lay mot mau that lam vi du: phai gui du ca 30 cot
mau = load_breast_cancer(as_frame=True).data.iloc[0].to_dict()
r = requests.post('http://127.0.0.1:5000/predict', json=mau)
print(r.json())      # nhan 1 la ac tinh; xem xac_suat_tung_lop trong ket qua

# Gui thieu cot thi API bao loi ro rang, khong doan bua
r = requests.post('http://127.0.0.1:5000/predict',
                  json={'mean radius': 17.99})
print(r.status_code, r.json())   # 400, kem danh sach cot thieu


Có ba điều cần nhớ khi đưa mô hình lên API. Trước hết, tệp mô hình phải chứa CẢ pipeline tiền xử lý, không chỉ mỗi bộ phân lớp — nếu không, dữ liệu gửi tới sẽ không được chuẩn hóa giống lúc huấn luyện và kết quả sai hoàn toàn.

Cần sắp xếp các cột theo đúng thứ tự khi huấn luyện. Không nên dựa vào thứ tự khóa trong JSON, vì pandas tạo DataFrame theo thứ tự nhận được. Mô hình huấn luyện trên mảng số thường đối chiếu đặc trưng theo vị trí, nên thay đổi thứ tự cột có thể làm kết quả sai mà không báo lỗi. Để phòng tránh, huấn luyện trên DataFrame, lưu tên cột qua feature_names_in_ và sắp xếp đầu vào theo danh sách đó bằng X = pd.DataFrame([du_lieu])[TEN_COT].

Kiểm tra đầu vào là đối tượng JSON, đủ cột và có giá trị số hữu hạn; nếu không hợp lệ, trả mã 400 cùng mô tả lỗi. Endpoint /health xác nhận tiến trình đang đáp ứng, không thay thế kiểm thử chất lượng dự đoán. Máy chủ Flask ở ví dụ dùng cho thực hành cục bộ; triển khai thật cần cấu hình phục vụ phù hợp.

### Bảng tổng kết các kỹ thuật đã thực hành

Bảng 6.8. Tổng kết các kỹ thuật đánh giá và tối ưu

### Nhận xét tổng hợp

Trên Ljubljana, cây quyết định cho thấy ba điều mà một con số Accuracy đơn lẻ không nói ra. Thứ nhất, Accuracy 73,08% đi kèm Recall 0,282 ở lớp tái phát: mô hình bỏ sót 61 trong 85 ca, nên phải đọc chỉ số theo từng lớp và theo hậu quả của từng loại sai lầm. Thứ hai, cấu hình mặc định của DecisionTreeClassifier không cắt tỉa, khoảng chênh huấn luyện – kiểm chứng chéo lên tới 33,07 điểm phần trăm; hạn chế độ phức tạp thu khoảng chênh về 3,60 điểm và đồng thời tăng điểm kiểm chứng chéo 8,07 điểm. Thứ ba, mức cắt tỉa tốt nhất theo Accuracy (0,01) lại không phải mức tốt nhất theo Recall (0,005), nên tiêu chí lựa chọn phải được chốt trước khi so sánh.

Các phần từ 6.4.D trở đi chuyển sang Wisconsin với lớp dương là ác tính; điểm cụ thể phải lấy từ lần chạy đúng cấu hình và đúng phiên bản thư viện. Quy trình chia tập, lựa chọn bằng kiểm chứng chéo và đánh giá một lần trên tập kiểm tra độc lập có thể áp dụng cho bài toán khác, nhưng mức điểm không chuyển nguyên xi giữa các tập dữ liệu.

### Bài thực hành tự làm

- Yêu cầu: Áp dụng toàn bộ quy trình 6.4.A đến 6.4.G trên một tập dữ liệu thực tế khác

- Gợi ý tập dữ liệu:

Heart Disease (UCI): bài toán phân lớp nhị phân y tế

- Pima Indians Diabetes: bài toán phân lớp nhị phân, dữ liệu mất cân bằng

- Adult Income (UCI): bài toán dự đoán thu nhập, tập dữ liệu lớn hơn. Báo cáo cần có:

- (a) Giới thiệu tập dữ liệu: nguồn, số mẫu, số đặc trưng, nhãn mục tiêu, mức độ cân bằng

- (b) Đánh giá mô hình ban đầu (chưa tối ưu) với đầy đủ chỉ số: Accuracy, Precision, Recall,

F1, ROC AUC. Nếu đa lớp, dùng cả Macro F1 và Micro F1

- (c) Phát hiện quá khớp/thiếu khớp và áp dụng kỹ thuật xử lý phù hợp

- (d) Áp dụng chính quy hóa (nếu phù hợp với mô hình)

- (e) Tối ưu siêu tham số bằng ÍT NHẤT hai trong ba: Grid Search, Random Search, Bayesian

Optimization. So sánh kết quả và thời gian

- (f) Đóng gói thành pipeline, lưu mô hình, viết mã minh họa cách nạp mô hình ở môi trường

khác

- (g) Đánh giá mô hình cuối trên tập kiểm tra chưa từng dùng (hold-out). Báo cáo nộp: tệp PDF

10-15 trang, kèm Jupyter Notebook đầy đủ mã

